In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from dotenv import load_dotenv
import os

_ = load_dotenv()

In [3]:
from crewai import Agent, LLM
from crewai_tools import SerperDevTool
from crewai import Task
from crewai import Crew, Process

In [4]:
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

In [5]:
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
import requests

class SearchInput(BaseModel):
    search_query: str = Field(description="The search query string")

class StrictSerperSearchTool(BaseTool):
    name: str = "search_the_internet"
    description: str = "Search the internet for a given query. Only takes a single search_query string."
    args_schema: type[BaseModel] = SearchInput

    def _run(self, search_query: str) -> str:
        response = requests.post(
            "https://google.serper.dev/search",
            headers={
                "X-API-KEY": os.environ["SERPER_API_KEY"],
                "Content-Type": "application/json"
            },
            json={"q": search_query}
        )
        results = response.json()
        organic = results.get("organic", [])[:5]
        formatted = "\n".join(
            f"- {r.get('title', '')}: {r.get('link', '')} — {r.get('snippet', '')}"
            for r in organic
        )
        return formatted

search_tool = StrictSerperSearchTool()

In [6]:
groq_llm = LLM(
    model="groq/openai/gpt-oss-120b",
    temperature=0
)

researcher = Agent(
    role="Learning Resource Researcher",
    goal="Find high-quality, current learning resources for a given topic, "
         "appropriate to the learner's stated level.",
     backstory=(
        "You are a meticulous research specialist who has helped thousands "
        "of learners find the right courses, articles, and tutorials — from "
        "complete beginners to advanced practitioners. You are careful to "
        "match resource difficulty exactly to the learner's stated level: "
        "you never recommend resources that assume knowledge the learner "
        "doesn't yet have, and you never waste an advanced learner's time "
        "on material that's beneath them."
    ),
    llm=groq_llm,
    tools=[search_tool],
    max_iter=3,
    verbose=True
)

In [7]:
planner = Agent(
    role="Curriculum Planner",
    goal="Sequence a set of learning resources into a realistic, "
         "prerequisite-aware, week-by-week learning path that fits the "
         "learner's time budget, level, and goal.",
    backstory=(
        "You are an experienced curriculum designer who has built learning "
        "paths for thousands of learners at every stage — beginners, "
        "career-changers leveling up, and experts deepening niche skills. "
        "You are pragmatic about pacing for each level: beginners need more "
        "time on fundamentals than they think, while advanced learners can "
        "move quickly through material they've already got a foundation in. "
        "You always respect prerequisites, regardless of level."
    ),
    llm=groq_llm,
    max_iter=3,
    verbose=True
)

In [8]:
writer = Agent(
    role="Learning Path Writer",
    goal="Turn a structured weekly plan into a polished, motivating, "
         "human-readable learning path write-up.",
    backstory=(
        "You are a friendly, encouraging learning coach and skilled "
        "technical writer. You explain not just what a learner should do "
        "each week, but why the sequence makes sense given their goal and "
        "level. You write in clear, professional prose without relying on "
        "emojis or overly casual language to convey enthusiasm."
    ),
    llm=groq_llm,
    max_iter=3,
    verbose=True
)

In [9]:
reviewer = Agent(
    role="Curriculum Reviewer",
    goal="Critically evaluate whether a proposed learning path genuinely "
         "fits the learner's stated time budget, level, and goal — and "
         "approve it if it's reasonably realistic, even if imperfect.",
    backstory=(
        "You are a strict but fair curriculum reviewer. You check pacing "
        "realism, whether resources match the learner's level, and whether "
        "the plan actually serves their goal. This is a v1 plan, not a "
        "perfect one — you approve it unless there's a genuine, significant "
        "mismatch (wildly unrealistic pacing, resources far above or below "
        "the learner's level, or a plan that doesn't serve the stated goal "
        "at all). Minor imperfections are expected and should not block "
        "approval."
    ),
    llm=groq_llm,
    max_iter=3,
    verbose=True
)

In [10]:
from pydantic import BaseModel, Field
from typing import List, Literal

class PlanItem(BaseModel):
    week: int = Field(description="Which week this item falls in")
    resource: str = Field(description="A single resource name or URL for this week")
    objective: str = Field(description="What the learner should achieve this week")

class WeeklyPlan(BaseModel):
    items: List[PlanItem]

class CritiqueResult(BaseModel):
    verdict: Literal["approve", "replan", "research"] = Field(
        description="approve if the plan is good, replan if sequencing/pacing "
                    "is wrong but resources are fine, research if resources "
                    "themselves are poor or missing"
    )
    feedback: str = Field(description="Specific, actionable feedback")

In [18]:
research_task = Task(
    description=(
        "Find 5 high-quality, current learning resources for the topic "
        "'{topic}', appropriate for a '{level}' level learner. "
        "Consider that their goal is: {goal}. "
        "Do ONE search first. If it returns at least 3-4 solid, relevant "
        "results (structured courses, official docs, or reputable platforms "
        "— not forum threads or discussions), use those and stop. Only "
        "search again if the first results are clearly insufficient. "
        "Previous feedback to address (if any): {feedback}. "
        "Respond with ONLY a plain numbered list: title, URL, and one "
        "sentence summary per resource. No tables, no extra sections."
    ),
    expected_output="A plain numbered list of 5 resources, one line each: title, URL, one-sentence summary.",
    agent=researcher
)

In [19]:
plan_task = Task(
    description=(
        "Using the resources found by the researcher, create a week-by-week "
        "learning plan for '{topic}' that fits within {time_weeks} weeks, "
        "for a '{level}' level learner with the goal: {goal}. "
        "Previous feedback to address (if any): {feedback}. "
        "Output ONLY valid JSON, no other text, in this exact format: "
        '{{"items": [{{"week": 1, "resource": "...", "objective": "..."}}]}}'
    ),
    expected_output="Valid JSON matching the specified format, nothing else.",
    agent=planner,
    context=[research_task]
)

In [21]:
write_task = Task(
    description=(
        "Turn the weekly plan into a motivating write-up for the learner. "
        "Topic: '{topic}', level: '{level}', goal: {goal}, time: "
        "{time_weeks} weeks. Do not use emojis. Keep it concise: one short "
        "paragraph per week, no extra tables or tip sections."
    ),
    expected_output="A concise markdown write-up, roughly 300-400 words total.",
    agent=writer,
    context=[plan_task],
    cache=False
)

In [22]:
critique_task = Task(
    description=(
        "Critically evaluate the generated learning path write-up against "
        "the original constraints: topic '{topic}', level '{level}', goal "
        "{goal}, time budget {time_weeks} weeks. Check pacing realism, "
        "resource-level match, and goal alignment. "
        "Output ONLY valid JSON, no other text, in this exact format: "
        '{{"verdict": "approve", "feedback": "..."}} '
        '(verdict must be exactly one of: approve, replan, research)'
    ),
    expected_output="Valid JSON matching the specified format, nothing else.",
    agent=reviewer,
    context=[write_task, plan_task],
    cache=False
)

In [15]:
crew = Crew(
    agents=[researcher, planner, writer, reviewer],
    tasks=[research_task, plan_task, write_task, critique_task],
    process=Process.sequential,
    verbose=True
)

In [16]:
import asyncio

async def kickoff_with_retry(crew, inputs, max_retries=3, wait_seconds=40):
    for attempt in range(max_retries):
        try:
            return await crew.kickoff_async(inputs=inputs)
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < max_retries - 1:
                print(f"Rate limited, waiting {wait_seconds}s before retry...")
                await asyncio.sleep(wait_seconds)
            else:
                raise

result = await kickoff_with_retry(crew, inputs={
    "topic": "Machine Learning Engineering",
    "level": "beginner",
    "goal": "job hunting",
    "time_weeks": 8
}, max_retries=4, wait_seconds=60)

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1a0ac9e1-0672-43e0-9294-c2d91bdc5499                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find 5 high-quality, current learning resources for the topic 'Machine Learning Engineering',            │
│  appropriate for a 'beginner' level learner. Consider that their goal is: job hunting. Respond with ONLY a      │
│  plain numbered list: title, URL, and one sentence summary per resource. No tables, no extra sections, no 'how  │
│  to use these' commentary.                                                                                      │
│  ID: 86769a62-5bea-4335-b916-135ac23c28b1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Learning Resource Researcher                                                                            │
│                                                                                                                 │
│  Task: Find 5 high-quality, current learning resources for the topic 'Machine Learning Engineering',            │
│  appropriate for a 'beginner' level learner. Consider that their goal is: job hunting. Respond with ONLY a      │
│  plain numbered list: title, URL, and one sentence summary per resource. No tables, no extra sections, no 'how  │
│  to use these' commentary.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet                                                                                      │
│  Args: {'search_query': 'beginner machine learning engineering course job ready'}                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet executed with result: - What should I learn now as a beginner to become an job ...: https://www.reddit.com/r/learnprogramming/comments/10kdmxq/what_should_i_learn_now_as_a_beginner_to_become/ — You need a Ph.D or MS to get...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Curriculum Reviewer                                                                                     │
│                                                                                                                 │
│  Task: Critically evaluate the generated learning path write-up against the original constraints: topic         │
│  'Machine Learning Engineering', level 'beginner', goal job hunting, time budget 8 weeks. Check pacing          │
│  realism, resource-level match, and goal alignment. Output ONLY valid JSON, no other text, in this exact        │
│  format: {{"verdict": "approve", "feedback": "..."}} (verdict must be exactly one of: approve, replan,          │
│  research)                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Curriculum Reviewer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "verdict": "replan",                                                                                         │
│    "feedback": "The plan targets a beginner with an 8‑week time budget but packs several multi‑month programs   │
│  (Udacity Nanodegree, Coursera MLOps specialization, IBM Professional Certificate) into a single week each.     │
│  This pacing is unrealistic for a novice and the depth of the later resources (MLOps, production deployment)    │
│  is likely above the learner's current level. While the goal of job hunting is addressed through credentials    │
│  and portfolio work, the schedule needs to be stretched and resources better matched to a beginner’s capacity.  │
│  Recommend reducing the number of intensive courses, extending the timeline, or selecting more bite‑sized       │
│  beginner resources to ensure feasible progress."                                                               │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Critically evaluate the generated learning path write-up against the original constraints: topic         │
│  'Machine Learning Engineering', level 'beginner', goal job hunting, time budget 8 weeks. Check pacing          │
│  realism, resource-level match, and goal alignment. Output ONLY valid JSON, no other text, in this exact        │
│  format: {{"verdict": "approve", "feedback": "..."}} (verdict must be exactly one of: approve, replan,          │
│  research)                                                                                                      │
│  Agent: Curriculum Reviewer                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
  "verdict": "replan",
  "feedback": "The plan targets a beginner with an 8‑week time budget but packs several multi‑month programs (Udacity Nanodegree, Coursera MLOps specialization, IBM Professional Certificate) into a single week each. This pacing is unrealistic for a novice and the depth of the later resources (MLOps, production deployment) is likely above the learner's current level. While the goal of job hunting is addressed through credentials and portfolio work, the schedule needs to be stretched and resources better matched to a beginner’s capacity. Recommend reducing the number of intensive courses, extending the timeline, or selecting more bite‑sized beginner resources to ensure feasible progress."
}


In [23]:
import json

MAX_ITERATIONS = 3

async def run_learning_path(topic, level, goal, time_weeks):
    inputs = {
        "topic": topic, "level": level, "goal": goal,
        "time_weeks": time_weeks, "feedback": "none"
    }

    # First full pass: research -> plan -> write -> critique
    full_crew = Crew(
        agents=[researcher, planner, writer, reviewer],
        tasks=[research_task, plan_task, write_task, critique_task],
        process=Process.sequential,
        verbose=True
    )
    await kickoff_with_retry(full_crew, inputs, max_retries=4, wait_seconds=60)

    for iteration in range(1, MAX_ITERATIONS + 1):
        critique_data = json.loads(critique_task.output.raw)
        verdict = critique_data["verdict"]
        feedback = critique_data["feedback"]
        print(f"\n--- Iteration {iteration}: verdict = {verdict} ---")

        if verdict == "approve" or iteration == MAX_ITERATIONS:
            if verdict != "approve":
                print("Max iterations reached — accepting current version.")
            return write_task.output.raw

        inputs["feedback"] = feedback

        if verdict == "research":
            rerun_crew = Crew(
                agents=[researcher, planner, writer, reviewer],
                tasks=[research_task, plan_task, write_task, critique_task],
                process=Process.sequential,
                verbose=True
            )
        else:  # "replan"
            rerun_crew = Crew(
                agents=[planner, writer, reviewer],
                tasks=[plan_task, write_task, critique_task],
                process=Process.sequential,
                verbose=True
            )

        await kickoff_with_retry(rerun_crew, inputs, max_retries=4, wait_seconds=60)

    return write_task.output.raw

final_result = await run_learning_path(
    topic="Machine Learning Engineering",
    level="beginner",
    goal="job hunting",
    time_weeks=8
)
print(final_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6e8cc813-059a-4407-b98e-4d7deccfe2ea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find 5 high-quality, current learning resources for the topic 'Machine Learning Engineering',            │
│  appropriate for a 'beginner' level learner. Consider that their goal is: job hunting. Do ONE search first. If  │
│  it returns at least 3-4 solid, relevant results (structured courses, official docs, or reputable platforms —   │
│  not forum threads or discussions), use those and stop. Only search again if the first results are clearly      │
│  insufficient. Previous feedback to address (if any): none. Respond with ONLY a plain numbered list: title,     │
│  URL, and one sentence summary per resource. No tables, no extra sections.                                      │
│  ID: a46e8c25-a30d-4d92-a534-ff62400b230b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Learning Resource Researcher                                                                            │
│                                                                                                                 │
│  Task: Find 5 high-quality, current learning resources for the topic 'Machine Learning Engineering',            │
│  appropriate for a 'beginner' level learner. Consider that their goal is: job hunting. Do ONE search first. If  │
│  it returns at least 3-4 solid, relevant results (structured courses, official docs, or reputable platforms —   │
│  not forum threads or discussions), use those and stop. Only search again if the first results are clearly      │
│  insufficient. Previous feedback to address (if any): none. Respond with ONLY a plain numbered list: title,     │
│  URL, and one sentence summary per resource. No tables, no extra sections.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#9) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet                                                                                      │
│  Args: {'search_query': 'beginner machine learning engineering course'}                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet executed with result: - Machine Learning Engineer: https://www.coursera.org/career-academy/roles/machine-learning-engineer — Skills you'll need: Machine Learning, Python Programming, Artificial Intelligence, Algorithms, Te...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Curriculum Reviewer                                                                                     │
│                                                                                                                 │
│  Task: Critically evaluate the generated learning path write-up against the original constraints: topic         │
│  'Machine Learning Engineering', level 'beginner', goal job hunting, time budget 8 weeks. Check pacing          │
│  realism, resource-level match, and goal alignment. Output ONLY valid JSON, no other text, in this exact        │
│  format: {{"verdict": "approve", "feedback": "..."}} (verdict must be exactly one of: approve, replan,          │
│  research)                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Curriculum Reviewer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "verdict": "approve",                                                                                        │
│    "feedback": "The 8‑week plan aligns well with a beginner aiming for ML engineering jobs. The chosen          │
│  resources (Google Crash Course, Kaggle intro, DataCamp fundamentals, Coursera deployment) are appropriate for  │
│  the learner’s level, and the sequence builds from core concepts to a portfolio‑ready end‑to‑end project.       │
│  Pacing is ambitious but realistic for a motivated student, and the final week directly supports job‑hunting    │
│  with portfolio, resume, and interview prep. Minor intensity in weeks 6‑7 is acceptable; overall the plan       │
│  serves the stated goal."                                                                                       │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Critically evaluate the generated learning path write-up against the original constraints: topic         │
│  'Machine Learning Engineering', level 'beginner', goal job hunting, time budget 8 weeks. Check pacing          │
│  realism, resource-level match, and goal alignment. Output ONLY valid JSON, no other text, in this exact        │
│  format: {{"verdict": "approve", "feedback": "..."}} (verdict must be exactly one of: approve, replan,          │
│  research)                                                                                                      │
│  Agent: Curriculum Reviewer                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Iteration 2: verdict = approve ---
**Week 1 – Foundations**  
Begin with Google’s Machine Learning Crash Course (Modules 1‑4) to acquire the essential vocabulary—features, labels, loss, and the training loop—while setting up a Python/TensorFlow environment. Establishing this solid conceptual base early ensures every later experiment rests on a clear understanding of how models learn.

**Week 2 – Data Preparation & Simple Models**  
Continue with Modules 5‑8 of the same crash course, focusing on data cleaning, feature engineering, and training your first linear and regression models in TensorFlow. Mastering preprocessing now prevents the common “garbage‑in, garbage‑out” problem and gives you confidence in building reproducible pipelines.

**Week 3 – Hands‑On Kaggle Project**  
Apply the concepts to a real dataset through Kaggle’s Intro to Machine Learning. Conduct exploratory analysis, construct baseline models, and publish a notebook. This public artifact demonstrates your ability